# Parsing well.ca orders

There are a couple of approaches:
- Scrape the website, logged in with token; need to take screenshots -- a lot more work, but reusable
- Download PDFs and parse them
- Take screenshots manually and bulk parse them like the other receipts
- Modify app to support PDF as images -- a lot of front-end complexity or conver to jpeg in the background anyway

Since I'm already working with JPGs and want JPGs to be able to use the AI, I'll just convert the PDFs to jpg then parse them like the other ones. It's a little counter intuitive to take text data to turn it into raster to then OCR, but it's the quickest way with my current setup. 

In [4]:
# brew install poppler

In [ ]:
%pip install -q \
    pdf2image \
    Pillow

Note: you may need to restart the kernel to use updated packages.


In [9]:
import os
from pdf2image import convert_from_path
from PIL import Image

# Directory containing PDF files
pdf_dir = './raw'

# Iterate over all PDF files in the directory
for pdf_file in os.listdir(pdf_dir):
    if pdf_file.endswith('.pdf'):
        pdf_path = os.path.join(pdf_dir, pdf_file)
        
        # Convert PDF to a list of images
        images = convert_from_path(pdf_path, dpi=300)
        
        # Resize images to width while maintaining aspect ratio
        resized_images = [img.resize((2400, int(img.height * 2400 / img.width)), Image.LANCZOS) for img in images]
        
        # Combine images vertically
        total_height = sum(img.height for img in resized_images)
        combined_image = Image.new('RGB', (2400, total_height))
        
        y_offset = 0
        for img in resized_images:
            combined_image.paste(img, (0, y_offset))
            y_offset += img.height
        
        # Save the combined image
        combined_image.save(os.path.join('./receipts', pdf_file.replace('.pdf', '.jpg')))